## Pipeline Smoke Test

A lightweight end-to-end smoke test for the Medallion pipeline. Verifies that key Bronze and Silver tables exist and contain at least one row. Intended to run as a Databricks Job task after the main pipeline.

### 1. Imports and Table Lists

Load PySpark and define the Bronze and Silver tables to smoke-test. Each table is referenced by its fully-qualified name (catalog.schema.table).

In [ ]:
# Databricks notebook source
# Simple pipeline smoke test for Medallion pipeline.
# Intended to run as a Databricks Job task after the main pipeline.

from pyspark.sql import functions as F

# Key Bronze and Silver tables - use fully-qualified names (catalog.schema.table).
bronze_tables = [
    "capstone.bronze.diseases_chronic_indicators_raw",
    "capstone.bronze.diseases_drug_label_raw_text",
    "capstone.bronze.geography_population_raw",
    "capstone.bronze.insurance_benefits_raw",
    "capstone.bronze.insurance_plan_attributes_raw",
    "capstone.bronze.inventory_metadata_raw",
    "capstone.bronze.inventory_raw",
    "capstone.bronze.medication_reviews_raw",
    "capstone.bronze.openfda_exclusivity_raw",
    "capstone.bronze.openfda_package_raw",
    "capstone.bronze.openfda_patent_raw",
    "capstone.bronze.openfda_product_raw",
    "capstone.bronze.openfda_products_raw",
    "capstone.bronze.pharmacies_raw",
]

silver_tables = [
    "capstone.silver.dim_insurance",
    "capstone.silver.dim_pharmacy",
    "capstone.silver.fact_medication_reviews",
    "capstone.silver.location_county",
    "capstone.silver.population",
]

### 2. Helper Functions

Defines `table_exists` and `basic_smoke_check` to verify that each table exists and has at least one row. Raises an error if a table is missing or empty.

In [ ]:
def table_exists(table_name: str) -> bool:
    try:
        spark.table(table_name)
        return True
    except Exception:
        return False


def basic_smoke_check(table_name: str):
    if not table_exists(table_name):
        raise AssertionError(f"Expected table '{table_name}' does not exist.")

    df = spark.table(table_name)
    row_count = df.limit(1).count()
    if row_count == 0:
        raise AssertionError(f"Table '{table_name}' exists but appears to be empty.")

### 3. Run Smoke Checks

Iterates over Bronze and Silver tables, runs the smoke check for each, and raises an error if any table fails. Used as a gate in CI to ensure the pipeline produced valid output.

In [ ]:
failed = []
categories = [("bronze", bronze_tables), ("silver", silver_tables)]

for layer_name, tables in categories:
    for tbl in tables:
        print(f"[SMOKE] Checking {layer_name} table: {tbl}")
        try:
            basic_smoke_check(tbl)
            print(f"[OK] {layer_name} table passed smoke test: {tbl}")
        except AssertionError as e:
            print(f"[FAIL] {e}")
            failed.append(str(e))

if failed:
    raise AssertionError(
        "Pipeline smoke test failed for one or more tables:\n" + "\n".join(failed)
    )

print("[RESULT] Pipeline smoke test completed successfully.")